# 01 — Data Collection

End-to-end view of how the FOX vs NBC headline dataset was assembled. The actual
scraping is implemented in the project scripts — this notebook documents the
process, shows a small worked example, and reports the final dataset shape.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent
EXTERNAL = ROOT / 'data' / 'external'
RAW      = ROOT / 'data' / 'raw'

## 1. Source URL list

We started from `data/external/url_only_data.csv`, the URL list provided for the
project. URLs were partitioned into FOX and NBC by host string and saved to
`data/external/fox_news_urls.csv` (FOX) — NBC URLs were collected directly via
the NBC sitemap because the original list was FOX-heavy.

In [2]:
url_list = pd.read_csv(EXTERNAL / 'url_only_data.csv')
fox_urls = url_list[url_list['url'].str.contains('foxnews.com', na=False)]
nbc_urls = url_list[url_list['url'].str.contains('nbcnews.com', na=False)]
pd.DataFrame([
    {'split': 'all', 'rows': len(url_list)},
    {'split': 'foxnews.com', 'rows': len(fox_urls)},
    {'split': 'nbcnews.com', 'rows': len(nbc_urls)},
])

,split,rows
0,all,3805
1,foxnews.com,2000
2,nbcnews.com,1805


## 2. Scraping pipeline

Scraping is implemented in two scripts under `scripts/`:

- `scripts/scrape_fox.py` — fetches each FOX URL, parses with `BeautifulSoup`,
  extracts `title / subtitle / author / datetime_posted / topic`, and writes a
  timestamped CSV under `data/raw/fox_scraped/`.
- `scripts/scrape_nbc.py` — same idea for NBC, with NBC-specific selectors and
  fallbacks for live blogs.
- `scripts/combine_scraped_fox.py` — concatenates per-day FOX CSVs into the
  canonical `data/raw/fox_scraped_all.csv`.
- `scripts/backfill_nbc.py` — backfills missing authors and posting dates on
  the NBC dataset where the first-pass scrape produced NA values.

A minimal worked example below (no network call) shows the per-record schema
the scrapers produce.

In [3]:
schema_example = {
    'url':              'https://www.foxnews.com/lifestyle/jack-carrs-eisenhower-d-day',
    'topic':            'lifestyle',
    'title':            'Jack Carr recalls Gen. Eisenhower\'s D-Day memo about "great and noble undertaking"',
    'subtitle':         'New book pays tribute to D-Day veterans...',
    'author':           'John Doe',
    'datetime_posted':  '2024-06-06T08:00:00-04:00',
    'label':            'Fox News',
}
pd.DataFrame([schema_example]).T.rename(columns={0: 'value'})

,value
url,https://www.foxnews.com/lifestyle/jack-carrs-e...
topic,lifestyle
title,Jack Carr recalls Gen. Eisenhower's D-Day memo...
subtitle,New book pays tribute to D-Day veterans...
author,John Doe
datetime_posted,2024-06-06T08:00:00-04:00
label,Fox News


## 3. Final scraped datasets

Two canonical CSVs live in `data/raw/` after scraping + backfill:

| file | rows | rough class label |
| --- | --- | --- |
| `fox_scraped_all.csv` | ~2,000 | FOX |
| `nbc_scraped_all.csv` | ~1,800 | NBC |

These are the inputs to `02_eda.ipynb` and `03_modeling.ipynb`.

In [4]:
fox = pd.read_csv(RAW / 'fox_scraped_all.csv')
nbc = pd.read_csv(RAW / 'nbc_scraped_all.csv')
pd.DataFrame([
    {'source': 'FOX', 'rows': len(fox), 'columns': list(fox.columns)},
    {'source': 'NBC', 'rows': len(nbc), 'columns': list(nbc.columns)},
])

,source,rows,columns
0,FOX,2000,"[url, topic, title, subtitle, author, datetime..."
1,NBC,1805,"[url, topic, title, subtitle, author, datetime..."


In [5]:
def na_summary(df, name):
    s = df.isna().sum().rename('na_count').to_frame()
    s['rows'] = len(df)
    s.index.name = 'column'
    return s.assign(source=name).reset_index()

pd.concat([na_summary(fox, 'FOX'), na_summary(nbc, 'NBC')], ignore_index=True).pivot(index='column', columns='source', values='na_count').fillna(0).astype(int)

source,FOX,NBC
column,,
author,0,4
datetime_posted,0,5
label,0,0
subtitle,13,4
title,0,4
topic,0,0
url,0,0
